# Unifying algorithm animation — simple (seminar) version

Slide-format (16:9) version of `../unifying_simulation/plot_simulation.ipynb`, using the layout
`animation_figure_simple.svg` (8 x 4.5 in, rendered at 240 dpi = 1920 x 1080 px, blank space at the top for a slide title).

Panels: x-y trajectory; inputs (wind rose, circular basis); transform (goal, scale, axis ratio);
output (course direction); course vs. time with wind-type labels.

Data is read from `../unifying_simulation/unifying_sim_data.parquet` (generated by `run_simulation.ipynb` there).
Frames are written to `<option>_frames_test/`. By default only a few preview frames are rendered;
set `RENDER_ALL = True` in the last cell to render the full sequence.

In [ ]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 8
matplotlib.rcParams['text.usetex'] = True
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'dejavusans'

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import figurefirst as fifi
import pynumdiff
from braid_analysis.flymath import wrap_angle

from splitflow.ghost_tail import (
    setup_xy_ghost_plot,
    setup_time_ghost_plot,
    update_ghost_tail,
    update_current_point,
)

# macOS: let cairocffi find Homebrew's libcairo (SIP can strip DYLD_* env vars before the kernel starts)
if sys.platform == "darwin":
    os.environ.setdefault("DYLD_FALLBACK_LIBRARY_PATH", "/opt/homebrew/lib")
import cairosvg

In [ ]:
option = 'smooth'  # 'smooth', 'saccadic', 'original'
TAIL = 200
dt = 0.01

DATA_FILE = '../unifying_simulation/unifying_sim_data.parquet'
SVG_TEMPLATE = 'animation_figure_simple.svg'
DPI = 240  # 8 x 4.5 in template -> 1920 x 1080 px

OUT = Path(option + "_frames_test")  # _test suffix so committed frame folders aren't overwritten
OUT.mkdir(exist_ok=True)

In [ ]:
df = pd.read_parquet(DATA_FILE)

if option == 'saccadic':
    x_label = 'x_sac'
    y_label = 'y_sac'
    course_label = 'course_sac'
    course_x_label = 'course_x_sac'
    course_y_label = 'course_y_sac'
    angvel_label = 'angvel_sac'
elif option == 'smooth':
    df['course_x_smooth'], _ = pynumdiff.savgoldiff(df.course_x.values, dt, [3, 101, 101])
    df['course_y_smooth'], _ = pynumdiff.savgoldiff(df.course_y.values, dt, [3, 101, 101])
    df['course_smooth'] = np.arctan2(df['course_y_smooth'], df['course_x_smooth'])

    speed = 0.3
    vx = df['course_x_smooth']*speed
    vy = df['course_y_smooth']*speed
    
    x = np.cumsum(vx)*dt
    y = np.cumsum(vy)*dt

    df['x_smooth'] = x
    df['y_smooth'] = y

    x_label = 'x_smooth'
    y_label = 'y_smooth'
    course_label = 'course_smooth'
    course_x_label = 'course_x_smooth'
    course_y_label = 'course_y_smooth'
    angvel_label = 'angvel_smooth'
    
elif option == 'original':
    x_label = 'x'
    y_label = 'y'
    course_label = 'course'
    course_x_label = 'course_x'
    course_y_label = 'course_y'
    angvel_label = 'angvel'

In [ ]:
def plot_single_bar(ax, height, color, width=0.6, bottom=0, **kwargs):
    """
    Plot a single bar with no edge/outline.

    Parameters
    ----------
    ax      : matplotlib Axes
    height  : height of the bar
    color   : fill color (any matplotlib color spec)
    width   : bar width (default 0.6)
    bottom  : bar base y-value (default 0, useful for stacking)
    **kwargs: forwarded to ax.bar()

    Returns
    -------
    bar : BarContainer
    """
    bar = ax.bar(
        0, height,
        width=width,
        bottom=bottom,
        color=color,
        edgecolor="none",
        linewidth=0,
        **kwargs,
    )
    return bar

In [ ]:
def remove_jumps(arr, threshold):
    """
    Replace values with np.nan where the signal jumps by more than
    `threshold` relative to the previous sample.

    Parameters
    ----------
    arr       : 1D numpy array
    threshold : scalar — jumps larger than this (in absolute value) are removed

    Returns
    -------
    out : copy of arr with jump locations set to np.nan
    """
    out = arr.astype(float).copy()
    jumps = np.abs(np.diff(out)) > threshold
    out[1:][jumps] = np.nan
    return out

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from typing import List, Optional, Tuple


def circular_distribution_plot(
    directions: List[float],
    magnitudes: List[float],
    ax: Optional[plt.Axes] = None,
    bins: int = 36,
    radial_limit: Optional[float] = None,
    log_scale: bool = False,
    normalize: bool = False,
    figsize: Tuple[int, int] = (7, 7),
) -> plt.Axes:
    """
    Create a circular distribution (polar) plot for vector directions and magnitudes.

    Bar height encodes mean magnitude per bin. Bar darkness encodes count density
    (how many vectors fall in that bin), mapped from light gray (few) to black (many).
    Together these show both the strength and prevalence of vectors in each direction.

    Parameters
    ----------
    directions   : Sequence of direction angles in radians (-pi to pi).
    magnitudes   : Corresponding magnitudes for each direction vector.
    ax           : Existing polar Axes to draw into. If None, a new figure is created.
    bins         : Number of angular bins around the circle (default 36 -> 10 deg each).
    radial_limit : Upper limit of the radial axis. Applied after any transforms.
    log_scale    : If True, apply log1p transform to mean magnitudes before plotting.
    normalize    : If True, normalize mean magnitudes to sum to 1 across bins.
                   Useful for comparing distributions with different overall scales.
    figsize      : Figure size in inches, used only when ax is None.

    Returns
    -------
    ax           : The polar Axes object.

    Example
    -------
    >>> import numpy as np
    >>> rng = np.random.default_rng(42)
    >>> dirs = rng.uniform(-np.pi, np.pi, 200)
    >>> mags = rng.rayleigh(scale=5, size=200)
    >>> ax = circular_distribution_plot(dirs, mags)
    """
    directions = np.asarray(directions, dtype=float)
    magnitudes = np.asarray(magnitudes, dtype=float)

    if directions.shape != magnitudes.shape:
        raise ValueError("directions and magnitudes must have the same length.")

    # --- Binning ---
    radians = directions % (2 * np.pi)
    bin_edges = np.linspace(0, 2 * np.pi, bins + 1)
    bin_width = bin_edges[1] - bin_edges[0]

    bin_indices = np.clip(np.digitize(radians, bin_edges) - 1, 0, bins - 1)

    bin_total_mag = np.zeros(bins)
    bin_counts = np.zeros(bins, dtype=float)
    for idx, mag in zip(bin_indices, magnitudes):
        bin_total_mag[idx] += mag
        bin_counts[idx] += 1

    # Mean magnitude per bin (bar height); empty bins stay at 0
    bin_mean_mag = np.where(bin_counts > 0, bin_total_mag / bin_counts, 0.0)

    # --- Transforms ---
    if normalize and bin_mean_mag.sum() > 0:
        bin_mean_mag = bin_mean_mag / bin_mean_mag.sum()

    if log_scale:
        bin_mean_mag = np.log1p(bin_mean_mag)

    # --- Bar colours: count density mapped to gray (light = few, dark = many) ---
    count_norm = Normalize(vmin=0, vmax=bin_counts.max() if bin_counts.max() > 0 else 1)
    # Map normalised count to a gray value: 0 -> 0.85 (light), 1 -> 0.15 (dark)
    gray_values = 0.85 - 0.70 * count_norm(bin_counts)
    bar_colors = [(g, g, g, 1.0) for g in gray_values]

    # --- Axes ---
    if ax is None:
        _, ax = plt.subplots(figsize=figsize, subplot_kw={"projection": "polar"})

    ax.set_theta_zero_location("E")
    ax.set_theta_direction(1)

    # --- Bars ---
    bars = ax.bar(
        bin_edges[:-1],
        bin_mean_mag,
        width=bin_width,
        color=bar_colors,
        edgecolor="none",
        linewidth=0.4,
        align="edge",
    )

    # --- Clean up labels and ticks ---
    ax.set_xticklabels([])
    #ax.set_yticklabels([])
    ax.grid(color="grey", linestyle="--", linewidth=0.5, alpha=0.4)

    if radial_limit is not None:
        ax.set_ylim(0, radial_limit)

    ax.spines["polar"].set_visible(False)
    
    return ax

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def draw_polar_arrow(ax, direction_rad, length, thickness=2, color='black', alpha=1.0):
    """
    Draw an arrow on a polar axis.

    Parameters
    ----------
    ax          : matplotlib polar Axes
    direction_rad : float — arrow direction in radians (0 = right, pi/2 = up)
    length      : float — arrow length in data units (same as radial axis)
    thickness   : float — line/arrow width in points (default 2)
    color       : str or color — arrow color
    alpha       : float — opacity (0–1)
    """
    ax.annotate(
        "",
        xy=(direction_rad, length),        # arrowhead (tip)
        xytext=(direction_rad, 0),         # arrow base (origin)
        arrowprops=dict(
            arrowstyle="-|>",
            color=color,
            lw=thickness,
            mutation_scale=thickness * 4,  # scales the arrowhead proportionally
        ),
        alpha=alpha,
    )

In [ ]:
def build_layout():
    """(Re-)construct the figurefirst layout and return the axes handles.
    The layout must be re-created each frame (figurefirst writes the mpl figure into the SVG tree)."""
    layout = fifi.svg_to_axes.FigureLayout(
        SVG_TEMPLATE,
        autogenlayers=True,
        make_mplfigures=True,
        hide_layers=[],
    )
    plt.close("all")
    axes = {name: layout.axes[('animation', name)]
            for name in ['trajec', 'wind_hist', 'circle', 'goal', 'scale', 'axis_ratio', 'affine', 'course']}
    return layout, axes


def bar_axis_style(ax, yticks, yticklabels=None):
    ax.set_xlim(-0.5, 0.5)
    ax.set_yticks(yticks)
    if yticklabels is not None:
        ax.set_yticklabels(yticklabels)
    fifi.mpl_functions.adjust_spines(ax, ['left'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)


def render_frame(frame_idx: int):
    """Render one frame to OUT/frame_XXXX.png. Returns the png path."""
    layout, axes = build_layout()
    behavior = df['behavior'][frame_idx]

    # -------------------------------------------------------------------------
    # x-y trajectory
    ax = axes['trajec']
    x = df[x_label].values
    y = df[y_label].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="black", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="black", markersize=7),
    )
    ax.set_xlim(-0.02, 1.16)   # aspect of the template rect matches these limits
    ax.set_ylim(-0.07, 0.19)
    ax.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax, [])
    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)

    # -------------------------------------------------------------------------
    # Inputs: wind rose (past 1 s) + current wind direction
    ax = axes['wind_hist']
    last = frame_idx
    first = np.max([0, last - 100])
    circular_distribution_plot(df.wind_direction.values[first:last],
                               df.wind_speed.values[first:last],
                               bins=50, ax=ax, normalize=False, radial_limit=0.5)
    ax.set_yticks([0, 0.25, 0.5])
    ax.set_yticklabels([])
    draw_polar_arrow(ax, df.wind_direction.values[last], 0.5, thickness=2, color='green', alpha=1.0)

    # Inputs: circular basis
    ax = axes['circle']
    x = df['circular_basis_x'].values
    y = df['circular_basis_y'].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="blue", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="blue", markersize=7),
    )
    ax.set_xlim(-1.15, 1.15)
    ax.set_ylim(-1.15, 1.15)
    ax.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax, [])
    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)

    # -------------------------------------------------------------------------
    # Transform: goal (gamma)
    ax = axes['goal']
    color = 'brown' if behavior == 'explore' else 'green'
    ax.plot(0, df['goal'].values[frame_idx], 'o', markerfacecolor=color, markeredgecolor=color, markersize=5)
    ax.set_ylim(-np.pi, np.pi)
    bar_axis_style(ax, [-np.pi, -np.pi/2, 0, np.pi/2, np.pi], [r'$-\pi$', '', '$0$', '', r'$\pi$'])

    # Transform: scale (xi)
    ax = axes['scale']
    plot_single_bar(ax, df['scale'].values[frame_idx], 'black', width=0.6, bottom=0)
    ax.set_ylim(0, 1)
    bar_axis_style(ax, [0, 0.5, 1])

    # Transform: axis ratio (lambda)
    ax = axes['axis_ratio']
    plot_single_bar(ax, df['lambda'].values[frame_idx], 'black', width=0.6, bottom=0)
    ax.set_ylim(0, 1)
    bar_axis_style(ax, [0, 0.2, 0.4, 0.6, 0.8, 1.0])

    # -------------------------------------------------------------------------
    # Output: course direction
    ax = axes['affine']
    x = df[course_x_label].values
    y = df[course_y_label].values
    lc_xy, pt_xy = setup_xy_ghost_plot(
        ax, x, y,
        tail_length=TAIL,
        tail_kw=dict(color="magenta", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="magenta", markersize=7),
    )
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    fifi.mpl_functions.adjust_spines(ax, [])
    update_ghost_tail(lc_xy, x, y, frame_idx)
    update_current_point(pt_xy, x, y, frame_idx)

    # -------------------------------------------------------------------------
    # Course vs time (x limits match the wind-type label segments in the template: 0-28 s)
    ax = axes['course']
    t = df['time'].values
    v = remove_jumps(df[course_label].values, 3)
    lc_ts, pt_ts = setup_time_ghost_plot(
        ax, t, v,
        tail_length=TAIL,
        tail_kw=dict(color="magenta", linewidth=2, alpha_max=0.85),
        point_kw=dict(color="magenta", markersize=7),
    )
    ax.set_xlim(0, 28)
    ax.set_ylim(-np.pi, np.pi)
    ax.set_ylabel('Course (rad)')
    ax.set_xlabel('Time (s)')
    ax.set_yticks([-np.pi, 0, np.pi])
    ax.set_yticklabels([r'$-\pi$', '$0$', r'$\pi$'])
    ax.set_xticks([0, 5, 10, 15, 20, 25])
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     spine_locations={'left': 5, 'bottom': 5},
                                     tick_length=2.5,)
    update_ghost_tail(lc_ts, t, v, frame_idx)
    update_current_point(pt_ts, t, v, frame_idx)

    # -------------------------------------------------------------------------
    # Dynamic svg text
    item = layout.svgitems['text_behavior_input']
    if behavior == 'explore':
        item.text = 'arbitrary'
        item.style['fill'] = '#964B00'
    else:
        item.text = 'upwind'
        item.style['fill'] = '#008000'

    goal_oriented = behavior in ('surge', 'explore')
    layout.svgitems['text_searching'].style['fill-opacity'] = 0. if goal_oriented else 1.
    layout.svgitems['text_goal_oriented'].style['fill-opacity'] = 1. if goal_oriented else 0.

    for wind_type in ['laminar', 'unsteady', 'low', 'still']:
        layout.svgitems['text_' + wind_type].style['fill-opacity'] = 0.2
    layout.svgitems['text_' + df['wind_type'][frame_idx]].style['fill-opacity'] = 1

    layout.apply_svg_attrs()

    # -------------------------------------------------------------------------
    # Embed mpl figure into the SVG layout, write, rasterise, delete svg
    layout.append_figure_to_layer(layout.figures["animation"], "animation", cleartarget=True)
    svg_path = OUT / f"frame_{frame_idx:04d}.svg"
    layout.write_svg(str(svg_path))
    png_path = svg_path.with_suffix(".png")
    cairosvg.svg2png(url=str(svg_path), write_to=str(png_path), dpi=DPI)
    os.remove(svg_path)

    plt.close("all")
    return png_path

In [ ]:
# Preview: one frame per behavior / wind condition
from IPython.display import Image, display

PREVIEW_FRAMES = [100, 300, 700, 1300, 1900, 2500]  # explore, surge, laminar, unsteady, low, still
for i in PREVIEW_FRAMES:
    png_path = render_frame(i)
    print(f"  frame {i:04d} ({df['behavior'][i]}, {df['wind_type'][i]}) -> {png_path}")
display(Image(filename=str(OUT / f"frame_{PREVIEW_FRAMES[3]:04d}.png"), width=960))

In [ ]:
# Full sequence (~15 min). Set RENDER_ALL = True once the preview frames look right.
RENDER_ALL = False
STRIDE = 2   # render every 2nd sample (dt = 0.01 s -> 50 frames per simulated second)

if RENDER_ALL:
    N = df.shape[0]
    for i in range(0, N, STRIDE):
        png_path = render_frame(i)
        if i % 100 == 0:
            print(f"  frame {i:04d} -> {png_path}")
    print(f"\nDone. Frames written to {OUT.resolve()}/")
    print("Combine into a video with e.g.:")
    print(f"  ffmpeg -r 30 -pattern_type glob -i '{OUT}/*.png' -c:v libx264 -pix_fmt yuv420p out_{option}_simple.mp4")